In [1]:
import os
import glob
from collections import Counter

# 스크린샷 구조에 맞춘 라벨 폴더 경로들
label_dirs = [
    './train/labels',
    './valid/labels',
    './test/labels'
]

all_class_ids = []

print("🔍 데이터셋 라벨 분석을 시작합니다...\n")

for label_dir in label_dirs:
    # 폴더가 존재하는지 확인
    if not os.path.exists(label_dir):
        print(f"⚠️ 경로를 찾을 수 없습니다: {label_dir}")
        continue

    # 폴더 내의 모든 .txt 파일 가져오기
    txt_files = glob.glob(os.path.join(label_dir, '*.txt'))
    print(f"📁 {label_dir} 폴더 내 txt 파일 {len(txt_files)}개 스캔 중...")

    # 각 텍스트 파일을 열어서 첫 번째 숫자(ID)만 수집
    for txt_file in txt_files:
        with open(txt_file, 'r') as f:
            lines = f.readlines()
            for line in lines:
                parts = line.strip().split()
                if len(parts) > 0:
                    try:
                        class_id = int(parts[0]) # 첫 번째 데이터가 클래스 ID
                        all_class_ids.append(class_id)
                    except ValueError:
                        pass # 숫자가 아닌 잘못된 라벨 데이터는 무시

# --------------------------
# 결과 출력
# --------------------------
if all_class_ids:
    class_counts = Counter(all_class_ids)
    print("\n✅ 분석 완료! 현재 데이터셋에 존재하는 클래스 ID와 각각의 개수입니다:")
    print("-" * 40)
    
    # ID 번호순으로 정렬하여 출력
    for cls_id in sorted(class_counts.keys()):
        print(f" - 클래스 ID [{cls_id}] : {class_counts[cls_id]:,}개")
        
    print("-" * 40)
    print(f"총 발견된 고유 클래스 ID 종류: {len(class_counts)}개")
    print(f"총 발견된 바운딩 박스(객체) 수: {len(all_class_ids):,}개")
else:
    print("\n❌ 라벨 데이터를 찾지 못했습니다. 스크립트를 실행하는 폴더 위치를 확인해 주세요.")

🔍 데이터셋 라벨 분석을 시작합니다...

📁 ./train/labels 폴더 내 txt 파일 2634개 스캔 중...
📁 ./valid/labels 폴더 내 txt 파일 966개 스캔 중...
📁 ./test/labels 폴더 내 txt 파일 458개 스캔 중...

✅ 분석 완료! 현재 데이터셋에 존재하는 클래스 ID와 각각의 개수입니다:
----------------------------------------
 - 클래스 ID [0] : 816개
 - 클래스 ID [1] : 3,632개
 - 클래스 ID [2] : 398개
 - 클래스 ID [3] : 148개
 - 클래스 ID [4] : 31,641개
 - 클래스 ID [5] : 703개
 - 클래스 ID [6] : 263개
 - 클래스 ID [7] : 5,842개
 - 클래스 ID [8] : 2,278개
 - 클래스 ID [9] : 3,672개
 - 클래스 ID [10] : 1,363개
 - 클래스 ID [11] : 821개
----------------------------------------
총 발견된 고유 클래스 ID 종류: 12개
총 발견된 바운딩 박스(객체) 수: 51,577개


In [2]:
import os
import glob
import yaml
from collections import Counter

# 1. data.yaml 파일 경로 설정 (본인의 위치에 맞게 수정)
yaml_path = 'data.yaml' 

# 2. data.yaml에서 클래스 이름 자동으로 가져오기
with open(yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)
    class_names = data_config.get('names', [])

print(f"📋 데이터셋에서 감지된 클래스 명칭 목록: {class_names}\n")

# 3. 라벨 폴더들
label_dirs = ['./train/labels', './valid/labels', './test/labels']
name_counts = Counter()

print("🚀 데이터셋 상세 분석 시작...")

for label_dir in label_dirs:
    if not os.path.exists(label_dir): continue
    
    txt_files = glob.glob(os.path.join(label_dir, '*.txt'))
    
    for txt_file in txt_files:
        with open(txt_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    try:
                        idx = int(parts[0])
                        # 리스트 범위를 벗어나지 않게 체크하며 이름 매칭
                        if 0 <= idx < len(class_names):
                            name_counts[class_names[idx]] += 1
                    except ValueError:
                        continue

# 4. 결과 출력
print(f"{'클래스 이름':<20} | {'개수'}")
print("-" * 35)
for name in class_names:
    print(f"{name:<20} | {name_counts.get(name, 0):,}개")

print("-" * 35)
print(f"총 객체 수: {sum(name_counts.values()):,}개")

📋 데이터셋에서 감지된 클래스 명칭 목록: ['big bus', 'big truck', 'bus-l-', 'bus-s-', 'car', 'mid truck', 'small bus', 'small truck', 'truck-l-', 'truck-m-', 'truck-s-', 'truck-xl-']

🚀 데이터셋 상세 분석 시작...
클래스 이름               | 개수
-----------------------------------
big bus              | 816개
big truck            | 3,632개
bus-l-               | 398개
bus-s-               | 148개
car                  | 31,641개
mid truck            | 703개
small bus            | 263개
small truck          | 5,842개
truck-l-             | 2,278개
truck-m-             | 3,672개
truck-s-             | 1,363개
truck-xl-            | 821개
-----------------------------------
총 객체 수: 51,577개


In [3]:
import os
import glob
import yaml

# 1. 경로 설정
yaml_path = 'data.yaml'
label_dirs = ['./train/labels', './valid/labels', './test/labels']

# 2. 통합 매핑 규칙
NAME_TO_ID = {
    'big bus': 0, 'bus-l-': 0, 'bus-s-': 0, 'small bus': 0,          # Bus -> 0
    'big truck': 1, 'mid truck': 1, 'small truck': 1,                # Truck -> 1
    'truck-l-': 1, 'truck-m-': 1, 'truck-s-': 1, 'truck-xl-': 1,
    'car': 2                                                         # Car -> 2
}

def preprocess():
    print("🚀 데이터 전처리(라벨 통합)를 시작합니다...")
    
    # yaml 파일 로드
    with open(yaml_path, 'r') as f:
        config = yaml.safe_load(f)
        class_names = config.get('names', [])

    # 라벨 변환
    for label_dir in label_dirs:
        txt_files = glob.glob(os.path.join(label_dir, '*.txt'))
        for txt_file in txt_files:
            with open(txt_file, 'r') as f:
                lines = f.readlines()
            
            new_lines = []
            for line in lines:
                parts = line.strip().split()
                if not parts: continue
                
                old_id = int(parts[0])
                old_name = class_names[old_id]
                new_id = NAME_TO_ID.get(old_name)
                
                if new_id is not None:
                    parts[0] = str(new_id)
                    new_lines.append(" ".join(parts) + "\n")
            
            with open(txt_file, 'w') as f:
                f.writelines(new_lines)
    print("✅ 모든 라벨 파일이 3개 클래스(Bus, Truck, Car)로 통합되었습니다.")

    # 캐시 삭제
    print("🧹 캐시 파일 삭제 중...")
    for d in label_dirs:
        cache_file = os.path.join(d, 'labels.cache')
        if os.path.exists(cache_file):
            os.remove(cache_file)
            print(f" - {cache_file} 삭제 완료")

if __name__ == '__main__':
    preprocess()

🚀 데이터 전처리(라벨 통합)를 시작합니다...
✅ 모든 라벨 파일이 3개 클래스(Bus, Truck, Car)로 통합되었습니다.
🧹 캐시 파일 삭제 중...


In [5]:
import os
import glob
from collections import Counter

label_dirs = ['./train/labels', './valid/labels', './test/labels']
id_counts = Counter()

for label_dir in label_dirs:
    txt_files = glob.glob(os.path.join(label_dir, '*.txt'))
    for txt_file in txt_files:
        with open(txt_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if not parts: continue
                # 라벨 파일의 ID만 직접 수집
                id_counts[int(parts[0])] += 1

print(f"{'통합된 ID':<10} | {'개수'}")
print("-" * 20)
for i in sorted(id_counts.keys()):
    print(f"{i:<10} | {id_counts[i]:,}개")

통합된 ID     | 개수
--------------------
0          | 1,625개
1          | 18,311개
2          | 31,641개
